# C8 · Secuenciación, formato FASTQ y control de calidad

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/08_fastq_qc/08_fastq_fastqc.ipynb)

## Pregunta guía

Dos archivos FASTQ tienen el mismo número de lecturas, pero uno contiene adaptadores y deterioro de calidad. **¿Qué métricas importan y qué acción se justifica para la pregunta biológica?**

### Objetivos

- comparar Sanger, lecturas cortas y largas según aplicación;
- leer la estructura FASTQ;
- convertir Phred en probabilidad de error;
- resumir calidad, longitud y GC con Python;
- ejecutar e interpretar FastQC;
- distinguir alerta real, sesgo esperado y artefacto de muestra pequeña.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Diseño antes de control de calidad

La plataforma, tipo de lectura, profundidad y replicación dependen de la pregunta. “Más cobertura” no corrige una biblioteca sesgada, una muestra contaminada o un diseño sin replicación.

## 2. FASTQ y Phred

Cada registro ocupa cuatro líneas: encabezado `@`, secuencia, separador `+` y calidad. En Phred+33:

\[
Q = -10 \log_{10}(P_{error}) \quad ; \quad P_{error}=10^{-Q/10}
\]

Q20 ≈ 1 error por 100 bases; Q30 ≈ 1 por 1.000; Q40 ≈ 1 por 10.000, bajo el modelo de calidad reportado.

In [ ]:
import pandas as pd

pd.DataFrame({
    "Q": [10, 20, 30, 40],
    "P_error": [10 ** (-q / 10) for q in [10, 20, 30, 40]],
    "accuracy_expected": [1 - 10 ** (-q / 10) for q in [10, 20, 30, 40]],
})

## 3. Parser mínimo y validación

In [ ]:
from pathlib import Path

def read_fastq(path: Path):
    with path.open(encoding="utf-8") as handle:
        while True:
            header = handle.readline().rstrip("\n")
            if not header:
                break
            seq = handle.readline().rstrip("\n")
            plus = handle.readline().rstrip("\n")
            qual = handle.readline().rstrip("\n")
            if not header.startswith("@") or not plus.startswith("+") or len(seq) != len(qual):
                raise ValueError(f"Registro FASTQ inválido en {path}: {header}")
            yield header[1:].split()[0], seq, [ord(c) - 33 for c in qual]

for name in ["good.fastq", "adapter_low_quality.fastq"]:
    records = list(read_fastq(ROOT / "data/module08" / name))
    print(name, len(records), "lecturas", len(records[0][1]), "bp")

## 4. Perfil de calidad por posición

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for name in ["good.fastq", "adapter_low_quality.fastq"]:
    records = list(read_fastq(ROOT / "data/module08" / name))
    q = np.array([r[2] for r in records])
    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, q.shape[1] + 1), q.mean(axis=0))
    plt.axhline(30, linestyle="--", linewidth=1)
    plt.xlabel("Posición en la lectura")
    plt.ylabel("Phred medio")
    plt.title(name)
    plt.ylim(0, 45)
    plt.show()

In [ ]:
def gc_fraction(seq: str) -> float:
    return (seq.count("G") + seq.count("C")) / len(seq)

rows = []
for name in ["good.fastq", "adapter_low_quality.fastq"]:
    records = list(read_fastq(ROOT / "data/module08" / name))
    rows.append({
        "file": name,
        "reads": len(records),
        "mean_length": np.mean([len(r[1]) for r in records]),
        "mean_gc": np.mean([gc_fraction(r[1]) for r in records]),
        "mean_q": np.mean([q for _, _, qs in records for q in qs]),
    })
pd.DataFrame(rows)

## 5. FastQC

In [ ]:
import shutil, subprocess
if shutil.which("fastqc"):
    result = subprocess.run(["bash", "scripts/module08/run_fastqc.sh"], cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
else:
    print("FastQC no está instalado en este kernel. Use el ambiente completo o ejecute el script en terminal/HPC.")

Interprete los módulos en contexto: calidad por base, contenido por base, GC, longitud, duplicación, secuencias sobre-representadas y adaptadores. Un módulo “FAIL” no ordena automáticamente recortar; indica que debe investigar.

### Checkpoint

El archivo problemático conserva muchas bases con calidad alta al inicio, pero muestra adaptador y Q bajo al final. Proponga una acción y una validación posterior; no diga simplemente “FastQC falló”.

## Reto

Justifique plataforma, tipo de lectura, profundidad y criterios QC para: resecuenciación bacteriana, ensamblaje vegetal repetitivo y estudio de amplicones.